# Notebook Overview: Stage 1 artifacts build notebook.ipynb

## Objective
This notebook is used to preprocess transactional data and build reusable stage-1 artifacts for fast and scalable recommendation candidate generation.

## Data Sources
Key datasets/artifacts used in this notebook:
- main_data.csv
- item_catalog.csv
- category_embedding_lookup

## Core Workflow
- Clean and normalize raw transaction fields and date context.
- Build lookup mappings (item-category, item-name, category-embedding).
- Construct basket-level tables and mine association rules.
- Compute co-purchase, context-popularity, and user-history counters.
- Save all runtime artifacts for stage-1 serving.

## Expected Outputs
- Association rules, co-purchase counters, and context counters.
- User behavior counters and date-context lookup artifacts.
- Serialized stage-1 runtime files for candidate generation.

## Role in Recommendation Pipeline
Acts as a central artifact-building notebook that powers stage-1 candidate retrieval logic.



<!-- AUTO_CELL_EXPLANATION -->
### Cell 1: ## Cell-wise Explanation (Easy)
**Explanation:** This markdown cell describes section context `## Cell-wise Explanation (Easy)` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


## Cell-wise Explanation (Easy)
- Cell 1: Stage-1 artifact build objective.
- Cell 2-3: Imports and dependencies.
- Cell 4-5: Data/artifact path setup.
- Cell 6-7: Raw, catalog, lookup data load.
- Cell 8-9: Helper functions (normalize, parser, utility).
- Cell 10-11: Raw data cleaning and type conversion.
- Cell 12-13: Lookup mapping build (`item_to_category`, `item_to_name`).
- Cell 14-15: Basket table aggregation by transaction.
- Cell 16-17: Association rule mining (frequent itemsets + rules).
- Cell 18-19: Item co-purchase counter build.
- Cell 20-21: Context popularity counter build.
- Cell 22-23: User-item and user-category history counters.
- Cell 24-25: Date context lookup creation.
- Cell 26-27: Customer default timeslot mapping.
- Cell 28-29: All stage-1 artifacts save (`.pkl`).
- Cell 30: Empty/placeholder cell.

### Possible Outputs
- Rule/itemset stats and counter sizes.
- Multiple artifact save confirmations (`association_rules.pkl`, counters, lookups).

<!-- AUTO_CELL_EXPLANATION -->
### Cell 2: IMPORTS
**Explanation:** This markdown cell describes section context `IMPORTS` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


 IMPORTS


<!-- AUTO_CELL_EXPLANATION -->
### Cell 3: ### Cell 1: import json
**Explanation:** This markdown cell describes section context `### Cell 1: import json` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 1: import json
**Explanation:** এই cell-এ `import json` সম্পর্কিত logic execute হয়।

**Possible Output:** Direct output না-ও থাকতে পারে; cell logic অনুযায়ী data update হবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 4: import json
**Explanation:** This cell imports dependencies for the stage-1 artifacts build pipeline.

**Possible Output:** Usually no visible output unless warnings appear.


In [2]:

import json
import pickle
import re
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

<!-- AUTO_CELL_EXPLANATION -->
### Cell 5: PATHS
**Explanation:** This markdown cell describes section context `PATHS` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


 PATHS


<!-- AUTO_CELL_EXPLANATION -->
### Cell 6: ### Cell 2: DATA_DIR = Path(r"D:\recommendation_item_API\data")
**Explanation:** This markdown cell describes section context `### Cell 2: DATA_DIR = Path(r"D:\recommendation_item_API\data")` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 2: DATA_DIR = Path(r"D:\recommendation_item_API\data")
**Explanation:** এই cell-এ `DATA_DIR = Path(r"D:\recommendation_item_API\data")` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 7: DATA_DIR = Path(r"D:\recommendation_item_API\data")
**Explanation:** This cell executes logic starting with `DATA_DIR = Path(r"D:\recommendation_item_API\data")` in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output.


In [2]:

DATA_DIR = Path(r"D:\recommendation_item_API\data")

RAW_FILE = DATA_DIR / "main_data.csv"
ITEM_CATALOG_FILE = DATA_DIR / "item_catalog.csv"
CATEGORY_LOOKUP_FILE = DATA_DIR / "category_embedding_lookup_from_customer_history.csv"

ARTIFACT_DIR = DATA_DIR / "stage1_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw file exists:", RAW_FILE.exists())
print("Catalog exists:", ITEM_CATALOG_FILE.exists())
print("Category lookup exists:", CATEGORY_LOOKUP_FILE.exists())

Raw file exists: True
Catalog exists: True
Category lookup exists: True


<!-- AUTO_CELL_EXPLANATION -->
### Cell 8: LOAD DATA
**Explanation:** This markdown cell describes section context `LOAD DATA` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


LOAD DATA


<!-- AUTO_CELL_EXPLANATION -->
### Cell 9: ### Cell 3: raw_df = pd.read_csv(RAW_FILE)
**Explanation:** This markdown cell describes section context `### Cell 3: raw_df = pd.read_csv(RAW_FILE)` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 3: raw_df = pd.read_csv(RAW_FILE)
**Explanation:** এই cell-এ `raw_df = pd.read_csv(RAW_FILE)` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 10: raw_df = pd.read_csv(RAW_FILE)
**Explanation:** This cell loads data/artifacts required by the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output, table/dataframe render.


In [3]:

raw_df = pd.read_csv(RAW_FILE)
item_catalog_df = pd.read_csv(ITEM_CATALOG_FILE)
cat_lookup_df = pd.read_csv(CATEGORY_LOOKUP_FILE)

raw_df.columns = [c.strip() for c in raw_df.columns]
item_catalog_df.columns = [c.strip() for c in item_catalog_df.columns]
cat_lookup_df.columns = [c.strip() for c in cat_lookup_df.columns]

display(raw_df.head())
display(item_catalog_df.head())
display(cat_lookup_df.head())

,Transaction_ID,Customer_ID,Timestamp,Product_Category,Product_Name,Quantity,Price,Total_Amount,Day_Type,transactionId,...,isFestival,isRamadan,isEidPrep,isEid,season,timeSlot,dayType,dayOfWeek,hour,month
0,1,23561,2025-01-01 07:29:00,Dairy-Other,Herman Peanut Butter-340gm,1,154.46,154.46,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
1,1,23561,2025-01-01 07:29:00,Bakery-Bread,Pran Toast-250g,4,63.66,254.64,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
2,1,23561,2025-01-01 07:29:00,Beverage-Hot,Horlicks Classic Malt-1Kg Jar,1,329.90,329.90,Weekday,1,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,7,1
3,2,23569,2025-01-01 11:33:00,Pantry-DryFood,Lassa Special Semai-500g(Pkt),2,155.51,311.02,Weekday,2,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,11,1
4,2,23569,2025-01-01 11:33:00,Fruits-Fresh,Atta Fruits,2,46.66,93.32,Weekday,2,...,0,0,0,0,Winter,Morning,Weekday,Wednesday,11,1


,itemId,itemName,category,avgPrice,minPrice,maxPrice,totalRowsSeen,nameVariantCount,categoryVariantCount,reviewFlag
0,13,Lemon Bright Dish Wash 250ml B2 G1,Household-Kitchen,121.34,121.34,121.34,115,1,1,0
1,27,Pran Toast-250g,Bakery-Bread,63.66,63.66,63.66,244,1,1,0
2,109,Wheel 2in1 Clean & Fresh Powder-1Kg,Personal-Care-Cosmetics,587.26,587.26,587.26,65,1,1,0
3,125,Wheel Laundry Soap 125g,Personal-Care-Bath,122.60,122.60,122.60,43,1,1,0
4,129,Vim Bar Lemon-125gm,Household-Kitchen,73.72,73.72,73.72,144,1,1,0


,category,cat_emb_0,cat_emb_1,cat_emb_2,cat_emb_3,cat_emb_4,cat_emb_5,cat_emb_6,cat_emb_7,category_embedding
0,Bakery-Bread,30.500512,-0.312345,-1.336538,0.171504,1.265434,-0.221737,-2.422268,-1.024791,"[30.500512, -0.312345, -1.336538, 0.171504, 1...."
1,Beverage-Carbonated,30.793406,0.582054,2.320642,-1.194838,0.826971,-1.544423,1.249234,-0.971547,"[30.793406, 0.582054, 2.320642, -1.194838, 0.8..."
2,Beverage-Hot,29.132634,0.224027,0.067448,-0.667184,-0.053071,0.364866,0.603695,0.191561,"[29.132634, 0.224027, 0.067448, -0.667184, -0...."
3,Beverage-Juice,31.321391,0.378736,0.439785,-0.637943,0.784723,-1.583087,-2.383535,2.169788,"[31.321391, 0.378736, 0.439785, -0.637943, 0.7..."
4,Beverage-Water,22.344446,0.596863,0.331393,-0.024937,0.778449,0.414786,-0.005348,-0.207390,"[22.344446, 0.596863, 0.331393, -0.024937, 0.7..."


<!-- AUTO_CELL_EXPLANATION -->
### Cell 11: HELPERS
**Explanation:** This markdown cell describes section context `HELPERS` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


HELPERS


<!-- AUTO_CELL_EXPLANATION -->
### Cell 12: ### Cell 4: def normalize_text(x):
**Explanation:** This markdown cell describes section context `### Cell 4: def normalize_text(x):` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 4: def normalize_text(x):
**Explanation:** এই cell-এ `def normalize_text(x):` সম্পর্কিত logic execute হয়।

**Possible Output:** Direct output নেই; function/helper define হবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 13: def normalize_text(x):
**Explanation:** This cell defines helper function `normalize_text` for the stage-1 artifacts build pipeline.

**Possible Output:** No direct output; definitions are available for later cells.


In [4]:

def normalize_text(x):
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

def normalize_category(cat):
    if pd.isna(cat):
        return cat
    cat = normalize_text(cat)
    parts = [p.strip() for p in cat.split("-")]
    parts = [p.capitalize() if p else p for p in parts]
    return "-".join(parts)

def week_of_month(dt):
    return ((dt.day - 1) // 7) + 1

<!-- AUTO_CELL_EXPLANATION -->
### Cell 14: CLEAN RAW DATA
**Explanation:** This markdown cell describes section context `CLEAN RAW DATA` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


CLEAN RAW DATA


<!-- AUTO_CELL_EXPLANATION -->
### Cell 15: ### Cell 5: raw_df["itemId"] = pd.to_numeric(raw_df["itemId"], errors="coerce")
**Explanation:** This markdown cell describes section context `### Cell 5: raw_df["itemId"] = pd.to_numeric(raw_df["itemId"], erro...` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 5: raw_df["itemId"] = pd.to_numeric(raw_df["itemId"], errors="coerce")
**Explanation:** এই cell-এ `raw_df["itemId"] = pd.to_numeric(raw_df["itemId"], errors="coerce")` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 16: raw_df["itemId"] = pd.to_numeric(raw_df["itemId"], errors="coerce")
**Explanation:** This cell executes logic starting with `raw_df["itemId"] = pd.to_numeric(raw_df["itemId"], errors="coerce")` in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output, table/dataframe render.


In [5]:

raw_df["itemId"] = pd.to_numeric(raw_df["itemId"], errors="coerce")
raw_df["customerId"] = pd.to_numeric(raw_df["customerId"], errors="coerce")
raw_df["category"] = raw_df["category"].apply(normalize_category)
raw_df["itemName"] = raw_df["itemName"].apply(normalize_text)

raw_df["orderDate"] = pd.to_datetime(raw_df["orderDate"], dayfirst=True, errors="coerce")
raw_df = raw_df.dropna(subset=["itemId", "customerId", "orderDate"]).copy()

raw_df["itemId"] = raw_df["itemId"].astype(int)
raw_df["customerId"] = raw_df["customerId"].astype(int)
raw_df["weekOfMonth"] = raw_df["orderDate"].apply(week_of_month)

print("Cleaned raw shape:", raw_df.shape)
display(raw_df.head())

Cleaned raw shape: (16108, 33)


,Transaction_ID,Customer_ID,Timestamp,Product_Category,Product_Name,Quantity,Price,Total_Amount,Day_Type,transactionId,...,isRamadan,isEidPrep,isEid,season,timeSlot,dayType,dayOfWeek,hour,month,weekOfMonth
0,1,23561,2025-01-01 07:29:00,Dairy-Other,Herman Peanut Butter-340gm,1,154.46,154.46,Weekday,1,...,0,0,0,Winter,Morning,Weekday,Wednesday,7,1,1
1,1,23561,2025-01-01 07:29:00,Bakery-Bread,Pran Toast-250g,4,63.66,254.64,Weekday,1,...,0,0,0,Winter,Morning,Weekday,Wednesday,7,1,1
2,1,23561,2025-01-01 07:29:00,Beverage-Hot,Horlicks Classic Malt-1Kg Jar,1,329.90,329.90,Weekday,1,...,0,0,0,Winter,Morning,Weekday,Wednesday,7,1,1
3,2,23569,2025-01-01 11:33:00,Pantry-DryFood,Lassa Special Semai-500g(Pkt),2,155.51,311.02,Weekday,2,...,0,0,0,Winter,Morning,Weekday,Wednesday,11,1,1
4,2,23569,2025-01-01 11:33:00,Fruits-Fresh,Atta Fruits,2,46.66,93.32,Weekday,2,...,0,0,0,Winter,Morning,Weekday,Wednesday,11,1,1


<!-- AUTO_CELL_EXPLANATION -->
### Cell 17: LOOKUPS
**Explanation:** This markdown cell describes section context `LOOKUPS` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


 LOOKUPS


<!-- AUTO_CELL_EXPLANATION -->
### Cell 18: ### Cell 6: item_catalog_df["itemId"] = pd.to_numeric(item_catalog_df["itemId...
**Explanation:** This markdown cell describes section context `### Cell 6: item_catalog_df["itemId"] = pd.to_numeric(item_catalog_...` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 6: item_catalog_df["itemId"] = pd.to_numeric(item_catalog_df["itemId"], e...
**Explanation:** এই cell-এ `item_catalog_df["itemId"] = pd.to_numeric(item_catalog_df["itemId"], e...` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 19: item_catalog_df["itemId"] = pd.to_numeric(item_catalog_df["itemId"], errors="...
**Explanation:** This cell executes logic starting with `item_catalog_df["itemId"] = pd.to_numeric(item_catalog_df["itemId"], errors="...` in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output.


In [6]:

item_catalog_df["itemId"] = pd.to_numeric(item_catalog_df["itemId"], errors="coerce")
item_catalog_df = item_catalog_df.dropna(subset=["itemId", "category"]).copy()
item_catalog_df["itemId"] = item_catalog_df["itemId"].astype(int)
item_catalog_df["category"] = item_catalog_df["category"].apply(normalize_category)

item_to_category = dict(zip(item_catalog_df["itemId"], item_catalog_df["category"]))
item_to_name = dict(zip(item_catalog_df["itemId"], item_catalog_df["itemName"]))

cat_lookup_df["category"] = cat_lookup_df["category"].apply(normalize_category)
cat_lookup_df["category_embedding_vec"] = cat_lookup_df["category_embedding"].apply(json.loads)

category_to_vector = {
    row["category"]: np.array(row["category_embedding_vec"], dtype=float)
    for _, row in cat_lookup_df.iterrows()
}

print("item_to_category:", len(item_to_category))
print("category_to_vector:", len(category_to_vector))

item_to_category: 229
category_to_vector: 43


<!-- AUTO_CELL_EXPLANATION -->
### Cell 20: BASKET TABLE
**Explanation:** This markdown cell describes section context `BASKET TABLE` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


 BASKET TABLE


<!-- AUTO_CELL_EXPLANATION -->
### Cell 21: ### Cell 7: basket_df = (
**Explanation:** This markdown cell describes section context `### Cell 7: basket_df = (` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 7: basket_df = (
**Explanation:** এই cell-এ `basket_df = (` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 22: basket_df = (
**Explanation:** This cell transforms/aggregates data to construct features in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output, table/dataframe render.


In [7]:

basket_df = (
    raw_df
    .sort_values(["transactionId", "orderDate"])
    .groupby("transactionId")
    .agg({
        "customerId": "first",
        "orderDate": "first",
        "season": "first",
        "isHoliday": "max",
        "isFestival": "max",
        "timeSlot": "first",
        "weekOfMonth": "first",
        "itemId": lambda x: list(map(int, x.tolist()))
    })
    .reset_index()
)

basket_df.rename(columns={"itemId": "item_ids"}, inplace=True)

print("Basket shape:", basket_df.shape)
display(basket_df.head())

Basket shape: (2426, 9)


,transactionId,customerId,orderDate,season,isHoliday,isFestival,timeSlot,weekOfMonth,item_ids
0,1,23561,2025-01-01 07:29:00,Winter,0,0,Morning,1,"[7075, 27, 6814]"
1,2,23569,2025-01-01 11:33:00,Winter,0,0,Morning,1,"[25474, 15131, 13786, 7017, 6815, 31849, 13352]"
2,3,23527,2025-01-01 12:39:00,Winter,0,0,Afternoon,1,"[32441, 2306, 13922, 2364, 952, 30743, 32269]"
3,4,23433,2025-01-01 12:46:00,Winter,0,0,Afternoon,1,"[878, 704, 15104, 31328, 14964]"
4,5,23494,2025-01-01 13:21:00,Winter,0,0,Afternoon,1,"[15180, 2220, 2364, 13793, 708, 2043, 12655]"


<!-- AUTO_CELL_EXPLANATION -->
### Cell 23: ASSOCIATION RULES
**Explanation:** This markdown cell describes section context `ASSOCIATION RULES` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


ASSOCIATION RULES


<!-- AUTO_CELL_EXPLANATION -->
### Cell 24: ### Cell 8: basket_items_for_rules = basket_df["item_ids"].apply(
**Explanation:** This markdown cell describes section context `### Cell 8: basket_items_for_rules = basket_df["item_ids"].apply(` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 8: basket_items_for_rules = basket_df["item_ids"].apply(
**Explanation:** এই cell-এ `basket_items_for_rules = basket_df["item_ids"].apply(` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 25: basket_items_for_rules = basket_df["item_ids"].apply(
**Explanation:** This cell trains/fits model components in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output, table/dataframe render.


In [9]:

basket_items_for_rules = basket_df["item_ids"].apply(
    lambda x: [str(v) for v in sorted(set(x))]
).tolist()

te = TransactionEncoder()
basket_matrix = te.fit(basket_items_for_rules).transform(basket_items_for_rules)
basket_matrix_df = pd.DataFrame(basket_matrix, columns=te.columns_)

freq_items = fpgrowth(
    basket_matrix_df,
    min_support=0.005,
    use_colnames=True,
    max_len=3
)

freq_items["itemset_len"] = freq_items["itemsets"].apply(len)

print("Frequent itemsets shape:", freq_items.shape)
display(freq_items.head())

# গুরুত্বপূর্ণ, এখানে full freq_items দিতে হবে
rules_df = association_rules(
    freq_items,
    metric="confidence",
    min_threshold=0.10
)

# চাইলে পরে rule filter করতে পারো
rules_df["antecedent_len"] = rules_df["antecedents"].apply(len)
rules_df["consequent_len"] = rules_df["consequents"].apply(len)

rules_df = rules_df[
    (rules_df["antecedent_len"] >= 1) &
    (rules_df["consequent_len"] >= 1)
].copy()

rules_df["antecedents"] = rules_df["antecedents"].apply(lambda s: sorted(list(s)))
rules_df["consequents"] = rules_df["consequents"].apply(lambda s: sorted(list(s)))

print("Rules shape:", rules_df.shape)
display(rules_df.head())

Frequent itemsets shape: (704, 3)


,support,itemsets,itemset_len
0,0.040396,(6814),1
1,0.037510,(27),1
2,0.023495,(7075),1
3,0.087799,(13352),1
4,0.079555,(25474),1


Rules shape: (713, 16)


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecedent_len,consequent_len
0,[6814],[3831],0.040396,0.044930,0.006183,0.153061,3.406665,1.0,0.004368,1.127673,0.736197,0.078125,0.113218,0.145338,1,1
1,[3831],[6814],0.044930,0.040396,0.006183,0.137615,3.406665,1.0,0.004368,1.112733,0.739692,0.078125,0.101312,0.145338,1,1
2,[6814],[13352],0.040396,0.087799,0.007007,0.173469,1.975759,1.0,0.003461,1.103651,0.514655,0.057823,0.093916,0.126641,1,1
3,[6814],[14034],0.040396,0.098516,0.005359,0.132653,1.346512,1.0,0.001379,1.039358,0.268173,0.040123,0.037868,0.093523,1,1
4,[6814],[27],0.040396,0.037510,0.007832,0.193878,5.168648,1.0,0.006317,1.193975,0.840477,0.111765,0.162461,0.201334,1,1


<!-- AUTO_CELL_EXPLANATION -->
### Cell 26: CO PURCHASE COUNTER
**Explanation:** This markdown cell describes section context `CO PURCHASE COUNTER` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


CO PURCHASE COUNTER


<!-- AUTO_CELL_EXPLANATION -->
### Cell 27: ### Cell 9: item_pair_counter = Counter()
**Explanation:** This markdown cell describes section context `### Cell 9: item_pair_counter = Counter()` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 9: item_pair_counter = Counter()
**Explanation:** এই cell-এ `item_pair_counter = Counter()` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 28: item_pair_counter = Counter()
**Explanation:** This cell executes logic starting with `item_pair_counter = Counter()` in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output.


In [10]:

item_pair_counter = Counter()

for _, row in basket_df.iterrows():
    items = sorted(set(row["item_ids"]))
    n = len(items)

    for i in range(n):
        for j in range(i + 1, n):
            a = items[i]
            b = items[j]
            item_pair_counter[(a, b)] += 1
            item_pair_counter[(b, a)] += 1

print("Directional item pairs:", len(item_pair_counter))

Directional item pairs: 30024


<!-- AUTO_CELL_EXPLANATION -->
### Cell 29: CONTEXT POPULARITY COUNTER
**Explanation:** This markdown cell describes section context `CONTEXT POPULARITY COUNTER` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


CONTEXT POPULARITY COUNTER


<!-- AUTO_CELL_EXPLANATION -->
### Cell 30: ### Cell 10: context_item_counter = Counter()
**Explanation:** This markdown cell describes section context `### Cell 10: context_item_counter = Counter()` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 10: context_item_counter = Counter()
**Explanation:** এই cell-এ `context_item_counter = Counter()` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 31: context_item_counter = Counter()
**Explanation:** This cell executes logic starting with `context_item_counter = Counter()` in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output.


In [11]:

context_item_counter = Counter()

for _, row in raw_df.iterrows():
    context_key = (
        row["season"],
        int(row["isHoliday"]),
        int(row["isFestival"]),
        row["timeSlot"],
        int(row["weekOfMonth"])
    )
    context_item_counter[(context_key, int(row["itemId"]))] += 1

print("Context item pairs:", len(context_item_counter))

Context item pairs: 6591


<!-- AUTO_CELL_EXPLANATION -->
### Cell 32: USER HISTORY COUNTERS
**Explanation:** This markdown cell describes section context `USER HISTORY COUNTERS` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


USER HISTORY COUNTERS


<!-- AUTO_CELL_EXPLANATION -->
### Cell 33: ### Cell 11: user_item_counter = Counter()
**Explanation:** This markdown cell describes section context `### Cell 11: user_item_counter = Counter()` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 11: user_item_counter = Counter()
**Explanation:** এই cell-এ `user_item_counter = Counter()` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 34: user_item_counter = Counter()
**Explanation:** This cell executes logic starting with `user_item_counter = Counter()` in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output.


In [12]:

user_item_counter = Counter()
user_category_counter = Counter()

for _, row in raw_df.iterrows():
    user_id = int(row["customerId"])
    item_id = int(row["itemId"])
    category = row["category"]

    user_item_counter[(user_id, item_id)] += 1
    user_category_counter[(user_id, category)] += 1

print("User item history size:", len(user_item_counter))
print("User category history size:", len(user_category_counter))

User item history size: 7275
User category history size: 3115


<!-- AUTO_CELL_EXPLANATION -->
### Cell 35: DATE CONTEXT LOOKUP
**Explanation:** This markdown cell describes section context `DATE CONTEXT LOOKUP` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


 DATE CONTEXT LOOKUP


<!-- AUTO_CELL_EXPLANATION -->
### Cell 36: ### Cell 12: date_context_df = (
**Explanation:** This markdown cell describes section context `### Cell 12: date_context_df = (` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 12: date_context_df = (
**Explanation:** এই cell-এ `date_context_df = (` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 37: date_context_df = (
**Explanation:** This cell transforms/aggregates data to construct features in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output, table/dataframe render.


In [13]:

date_context_df = (
    raw_df
    .assign(orderDay=raw_df["orderDate"].dt.date.astype(str))
    .groupby("orderDay")
    .agg({
        "season": lambda x: x.mode().iat[0] if len(x.mode()) > 0 else x.iloc[0],
        "isHoliday": "max",
        "isFestival": "max"
    })
    .reset_index()
)

date_context_lookup = {
    row["orderDay"]: {
        "season": row["season"],
        "isHoliday": int(row["isHoliday"]),
        "isFestival": int(row["isFestival"])
    }
    for _, row in date_context_df.iterrows()
}

print("Date context entries:", len(date_context_lookup))
display(date_context_df.head())

Date context entries: 144


,orderDay,season,isHoliday,isFestival
0,2025-01-01,Winter,0,0
1,2025-01-02,Winter,1,0
2,2025-01-03,Summer,1,1
3,2025-01-04,Summer,0,1
4,2025-01-05,Summer,1,0


<!-- AUTO_CELL_EXPLANATION -->
### Cell 38: CUSTOMER DEFAULT TIMESLOT
**Explanation:** This markdown cell describes section context `CUSTOMER DEFAULT TIMESLOT` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


 CUSTOMER DEFAULT TIMESLOT


<!-- AUTO_CELL_EXPLANATION -->
### Cell 39: ### Cell 13: customer_default_timeslot_df = (
**Explanation:** This markdown cell describes section context `### Cell 13: customer_default_timeslot_df = (` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 13: customer_default_timeslot_df = (
**Explanation:** এই cell-এ `customer_default_timeslot_df = (` সম্পর্কিত logic execute হয়।

**Possible Output:** Console print বা dataframe preview দেখাবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 40: customer_default_timeslot_df = (
**Explanation:** This cell transforms/aggregates data to construct features in the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output, table/dataframe render.


In [14]:

customer_default_timeslot_df = (
    raw_df.groupby("customerId")["timeSlot"]
    .agg(lambda x: x.mode().iat[0] if len(x.mode()) > 0 else x.iloc[0])
    .reset_index()
)

customer_default_timeslot = dict(
    zip(customer_default_timeslot_df["customerId"], customer_default_timeslot_df["timeSlot"])
)

print("Customer default timeslot size:", len(customer_default_timeslot))
display(customer_default_timeslot_df.head())

Customer default timeslot size: 137


,customerId,timeSlot
0,4959,Evening
1,23380,Evening
2,23381,Morning
3,23383,Night
4,23385,Night


<!-- AUTO_CELL_EXPLANATION -->
### Cell 41: SAVE ARTIFACTS
**Explanation:** This markdown cell describes section context `SAVE ARTIFACTS` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders explanatory text in notebook view.


SAVE ARTIFACTS


<!-- AUTO_CELL_EXPLANATION -->
### Cell 42: ### Cell 14: rules_file = ARTIFACT_DIR / "association_rules.pkl"
**Explanation:** This markdown cell describes section context `### Cell 14: rules_file = ARTIFACT_DIR / "association_rules.pkl"` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 14: rules_file = ARTIFACT_DIR / "association_rules.pkl"
**Explanation:** এই cell-এ `rules_file = ARTIFACT_DIR / "association_rules.pkl"` সম্পর্কিত logic execute হয়।

**Possible Output:** Saved file path/message print হতে পারে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 43: rules_file = ARTIFACT_DIR / "association_rules.pkl"
**Explanation:** This cell saves generated outputs/artifacts from the stage-1 artifacts build pipeline.

**Possible Output:** Possible output: console/text output.


In [15]:

rules_file = ARTIFACT_DIR / "association_rules.pkl"
pair_file = ARTIFACT_DIR / "item_pair_counter.pkl"
context_file = ARTIFACT_DIR / "context_item_counter.pkl"
user_item_file = ARTIFACT_DIR / "user_item_counter.pkl"
user_cat_file = ARTIFACT_DIR / "user_category_counter.pkl"
item_to_cat_file = ARTIFACT_DIR / "item_to_category.pkl"
item_to_name_file = ARTIFACT_DIR / "item_to_name.pkl"
cat_vec_file = ARTIFACT_DIR / "category_to_vector.pkl"
date_ctx_file = ARTIFACT_DIR / "date_context_lookup.pkl"
cust_ts_file = ARTIFACT_DIR / "customer_default_timeslot.pkl"

with open(rules_file, "wb") as f:
    pickle.dump(rules_df, f)

with open(pair_file, "wb") as f:
    pickle.dump(item_pair_counter, f)

with open(context_file, "wb") as f:
    pickle.dump(context_item_counter, f)

with open(user_item_file, "wb") as f:
    pickle.dump(user_item_counter, f)

with open(user_cat_file, "wb") as f:
    pickle.dump(user_category_counter, f)

with open(item_to_cat_file, "wb") as f:
    pickle.dump(item_to_category, f)

with open(item_to_name_file, "wb") as f:
    pickle.dump(item_to_name, f)

with open(cat_vec_file, "wb") as f:
    pickle.dump(category_to_vector, f)

with open(date_ctx_file, "wb") as f:
    pickle.dump(date_context_lookup, f)

with open(cust_ts_file, "wb") as f:
    pickle.dump(customer_default_timeslot, f)

print("Stage 1 artifacts saved.")

Stage 1 artifacts saved.


<!-- AUTO_CELL_EXPLANATION -->
### Cell 44: ### Cell 15: Code cell execution step
**Explanation:** This markdown cell describes section context `### Cell 15: Code cell execution step` for the stage-1 artifacts build pipeline.

**Possible Output:** Renders headings and bullet points in notebook view.


### Cell 15: Code cell execution step
**Explanation:** এই cell-এ `Code cell execution step` সম্পর্কিত logic execute হয়।

**Possible Output:** Direct output না-ও থাকতে পারে; cell logic অনুযায়ী data update হবে।

<!-- AUTO_CELL_EXPLANATION -->
### Cell 45: Code cell execution step
**Explanation:** This cell is a placeholder in the stage-1 artifacts build pipeline.

**Possible Output:** May execute silently with variable/state updates.
